### RAG-Powered Chatbot for University of Chicago Applied Data Science Program Questions
### Built with LangChain, OpenAI, and Chroma

This notebook builds a question-answering chatbot that scrapes the official UChicago MS in Applied Data Science website and stores the content in a searchable database. When a user asks a question, the chatbot retrieves the most relevant passages from that database and passes them to a language model to generate a grounded, source-cited answer. Personal information is automatically redacted from all outputs, and the system is designed to refuse answering questions that fall outside the scope of the official program pages.

In [1]:
# This cell loads all the external tools and libraries the project depends on, and sets up the folder structure where data will be stored.
# OpenAI was chosen as the AI provider because it offers strong pre-built embedding and chat models that integrate easily with LangChain.
import os, re, time, glob, uuid, urllib.parse, json, textwrap, requests
from bs4 import BeautifulSoup

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate

# The API key is read from the system environment rather than written directly in the code, so it is never accidentally shared or published.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

DATA_DIR = "msads_data"
HTML_DIR = f"{DATA_DIR}/html"
TEXT_DIR = f"{DATA_DIR}/text"
INDEX_DIR = f"{DATA_DIR}/index"
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(TEXT_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"
K_RETRIEVE  = 5
CHUNK_SIZE  = 1000
CHUNK_OVERLAP = 200

In [3]:
# This cell downloads the raw HTML content from a fixed list of UChicago MS-ADS web pages and saves each one as a file on disk.
# A fixed list of URLs was used instead of a web crawler because the relevant pages are known in advance, which avoids accidentally downloading unrelated content.

SEED = "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"

TARGET_URLS = [
    SEED,
    "https://datascience.uchicago.edu/education/masters-programs/in-person-program/",
    "https://datascience.uchicago.edu/education/masters-programs/online-program/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/our-students/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff/",
    "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs/",
]

def download_all(urls=TARGET_URLS, out_dir=HTML_DIR, throttle=0.3):
    """
    Downloads each page in TARGET_URLS and saves raw HTML into HTML_DIR.
    """
    headers = {"User-Agent": "UChicago-MSADS-RAG (class project)"}
    os.makedirs(out_dir, exist_ok=True)
    for url in TARGET_URLS:
        try:
            resp = requests.get(url, headers=headers, timeout=20)
            resp.raise_for_status()
            fname = urllib.parse.quote(url, safe="") + ".html"
            with open(os.path.join(out_dir, fname), "w", encoding="utf-8") as f:
                f.write(resp.text)
            print(f"Saved: {url}")
        except Exception as e:
            print(f"Skipped {url}: {e}")
    print(f"{len(urls)} pages saved in {out_dir}")

download_all()

Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/
Saved: https://datascience.uchicago.edu/education/masters-programs/in-person-program/
Saved: https://datascience.uchicago.edu/education/masters-programs/online-program/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/our-students/
Saved: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/instructors-staff/
Saved: https://datascience.uchicago.edu/education/masters-programs/

In [4]:
# This cell strips away the navigation menus, footers, and other boilerplate from each downloaded web page, keeping only the meaningful text content.
# Removing this clutter before storing text means the AI will only search through content that is actually relevant to a user's question.

DROP_SELECTORS = ["nav", "footer", ".site-footer", ".menu", ".breadcrumbs", "script", "style"]

def clean_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for sel in DROP_SELECTORS:
        for tag in soup.select(sel):
            tag.decompose()
    text = soup.get_text("\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text).strip()
    return text

def run_clean(in_dir=HTML_DIR, out_dir=TEXT_DIR):
    for fn in glob.glob(f"{in_dir}/*.html"):
        with open(fn, "r", encoding="utf-8") as f:
            html = f.read()
        txt = clean_to_text(html)
        base = os.path.basename(fn).replace(".html", "")
        with open(f"{out_dir}/{base}.txt", "w", encoding="utf-8") as f:
            f.write(txt)
    print("Cleaned:", len(glob.glob(f'{out_dir}/*.txt')), "files")

run_clean()

Cleaned: 10 files


In [5]:
# This cell breaks the cleaned text into small overlapping chunks and converts them into numerical representations called embeddings, which are then saved in a searchable database called Chroma.
# Chroma was chosen over FAISS because it automatically saves the index to disk, so the embeddings do not need to be rebuilt every time the notebook is reopened.

from pathlib import Path
from langchain_community.vectorstores import Chroma

def load_docs(txt_dir=TEXT_DIR):
    docs = []
    for path in glob.glob(f"{txt_dir}/*.txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        url = urllib.parse.unquote(os.path.basename(path).replace(".txt","").replace("%2F","/"))
        docs.append({"text": text, "source": url})
    return docs

def chunk_docs(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=["\n\n", "\n", ". ", " "]
    )
    texts, metas = [], []
    for d in docs:
        parts = splitter.split_text(d["text"])
        for i, p in enumerate(parts):
            texts.append(p)
            metas.append({"source": d["source"], "chunk_id": i})
    return texts, metas

PERSIST_DIR = f"{INDEX_DIR}/chroma"
Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)

def build_chroma():
    docs = load_docs()
    texts, metas = chunk_docs(docs)
    if not texts:
        raise RuntimeError("No chunks created -- rerun cleaning or lower filters.")
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    vs = Chroma.from_texts(
        texts=texts,
        embedding=embeddings,
        metadatas=metas,
        persist_directory=PERSIST_DIR,
        collection_name="msads_msads"
    )
    vs.persist()
    print(f"Built Chroma index with {len(texts)} chunks at {PERSIST_DIR}")

build_chroma()

Built Chroma index with 307 chunks at msads_data/index/chroma


/var/folders/16/c9ry1rxn4l7fqtv12zpchyqm0000gn/T/ipykernel_13619/76826348.py:45: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vs.persist()


In [6]:
# This cell defines two functions that automatically detect and remove personal information such as email addresses and phone numbers from any text before it is shown to the user.
# Regular expressions were used here because they can match many different formats of emails and phone numbers with a single pattern, making the scrubber robust without needing a long list of special cases.

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"\+?\d[\d\-\(\) ]{7,}\d")

def redact_pii(s: str) -> str:
    s = EMAIL_RE.sub("[REDACTED_EMAIL]", s)
    s = PHONE_RE.sub("[REDACTED_PHONE]", s)
    return s

In [7]:
# This cell defines the instructions that tell the AI how to behave when answering questions, including rules like only using information from the official MS-ADS pages and always listing its sources.
# Constraining the AI to the provided context rather than its general knowledge prevents it from inventing program details that may be outdated or incorrect.

SYSTEM_PROMPT = """You are a helpful assistant for the University of Chicago MS in Applied Data Science.
Answer ONLY using the provided context. If the answer is not in the context, say
'I don't know based on the official MS-ADS pages I have.' Keep answers concise,
quote or paraphrase key lines, and include source URLs."""

HUMAN_PROMPT = """Question:
{question}

Context:
{context}

Rules:
- Do not invent program details or deadlines.
- Prefer bullet points for lists.
- End with 'Sources:' and list the source URLs you used.
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT), ("human", HUMAN_PROMPT)]
)

In [8]:
# This cell sets up the retrieval system, which finds the most relevant text chunks from the database for a given question, and adds a multi-query step that rephrases each question three ways before searching.
# Rephrasing the question three ways before searching increases the chance of finding relevant content, because different phrasings may match different parts of the stored text.

from langchain_core.documents import Document

emb = OpenAIEmbeddings(model=EMBED_MODEL)
vs = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=emb,
    collection_name="msads_msads"
)
retriever = vs.as_retriever(search_kwargs={"k": K_RETRIEVE})

llm_qt = ChatOpenAI(model=CHAT_MODEL, temperature=0)

QT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You rewrite questions into diverse, relevant search queries for semantic retrieval."),
    ("human", "Rewrite the user question into 3 diverse, concise search queries focused on MS-ADS site.\n\nQuestion: {q}")
])

def multi_query(q: str):
    out = llm_qt.invoke(QT_PROMPT.format_messages(q=q))
    lines = [l.strip("-* ").strip() for l in out.content.split("\n") if l.strip()]
    if len(lines) < 2:
        lines = [q, f"What is {q} (MS-ADS)?", f"Details about {q} on MS-ADS"]
    return lines[:3]

def retrieve_with_qt(q: str, k=K_RETRIEVE):
    queries = multi_query(q)
    hits = []
    for mq in queries:
        docs = retriever.invoke(mq) or []
        hits.extend(docs)
    seen = set(); uniq = []
    for h in hits:
        key = (h.metadata.get("source",""), h.metadata.get("chunk_id",-1))
        if key not in seen:
            seen.add(key); uniq.append(h)
        if len(uniq) >= k:
            break
    return uniq

/var/folders/16/c9ry1rxn4l7fqtv12zpchyqm0000gn/T/ipykernel_13619/2089976590.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vs = Chroma(


In [9]:
# This cell assembles the full pipeline that takes a user question, retrieves the relevant text, and passes everything to the AI model to generate a final answer.
# LangChain's pipeline structure was used because it keeps each step modular, making it easy to swap out the retriever or the language model later without rewriting the whole flow.

def format_docs(docs):
    blocks = []
    for d in docs:
        src = d.metadata.get("source","")
        blocks.append(f"[Source: {src}]\n{redact_pii(d.page_content)}")
    return "\n\n---\n\n".join(blocks)

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

def build_chain():
    def _retrieve(inputs):
        q = redact_pii(inputs["question"])
        docs = retrieve_with_qt(q, k=K_RETRIEVE)
        return {"question": q, "context": format_docs(docs), "docs": docs}

    chain = (
        RunnablePassthrough.assign(**{"question": lambda x: x["question"]})
        | _retrieve
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    return chain

rag_chain = build_chain()

In [10]:
# This cell defines the answer function, which runs a question through the full pipeline and returns the AI's response along with the source pages it drew from.
# A hallucination check was added so that if no relevant documents are found, the chatbot returns a safe default message rather than generating an answer from nothing.

def answer(question: str):
    docs = retrieve_with_qt(question, k=K_RETRIEVE)
    ctx = format_docs(docs)
    msg = rag_prompt.format(question=redact_pii(question), context=ctx)
    out = llm.invoke(msg).content

    urls = list({d.metadata.get("source","") for d in docs if d.metadata.get("source")})
    urls = [u if u.startswith("http") else urllib.parse.unquote(u) for u in urls]
    if "Sources:" not in out:
        out += "\n\nSources:\n" + "\n".join(urls)
    elif out.strip().endswith("Sources:"):
        out += "\n" + "\n".join(urls)

    if not docs:
        out = "I don't know based on the official MS-ADS pages I have.\n\nSources:\n( none )"
    return out, docs

In [26]:
# This cell runs a sample question through the chatbot to verify that the full pipeline is working correctly from retrieval through to the final answer.
# Testing with a known question about faculty is useful because the answer can be visually verified against the actual MS-ADS website.

test_question = "Who are the faculty of the Applied Data Science program?"

try:
    answer_text, retrieved_docs = answer(test_question)
except Exception as e:
    raise RuntimeError(f"RAG pipeline error: {e}")

print("QUESTION:\n", test_question, "\n")
print("ANSWER:\n", answer_text, "\n")

unique_sources = []
seen = set()
for d in retrieved_docs:
    src = d.metadata.get("source", "")
    if src and src not in seen:
        seen.add(src)
        unique_sources.append(src)

QUESTION:
 Who are the faculty of the Applied Data Science program? 

ANSWER:
 The faculty of the Applied Data Science program includes:

- **Greg Green**: Senior Instructional Professor; Senior Director of the DSI Polsky Transform Initiative; Senior Director of DSI Executive and Professional Education.
- **Arnab Bose, PhD**: Faculty member (specific role not detailed).
- **DuJuan Smith, PhD**: Director, Student Affairs.
- **Alison Ossyra**: Director, External Partnerships and Career Services.
- **Brody Tate, EdD**: Program Manager, Online.
- **Daniel Truesdale, MPP**: Director, Enrollment Management and Analytics.
- **Taylor Wallace, MEd**: Graduate Academic Advisor.
- **Samantha Widemon, MNA**: Graduate Academic Advisor.
- **Jennifer Wei, MEM**: Assistant Director, Career Services.
- **Patrick Vonesh**: Senior Assistant Director, Enrollment Management.
- **Henry Igunbor**: Senior Manager of Facilities and IT.
- **Kendall Cox**: Operations Coordinator.
- **Zach Brown**: Operations Coo